# 07 — Investigation 1: The Advection Cliff

**Question**: PINN training collapses at high advection speeds. Why?
Is the network *unable to represent* the solution, or *unable to find* it?

**Protocol**:
1. Train PINN via physics loss at c = 1, 5, 20, 50 — measure where it collapses
2. **Supervised cheat test**: Fit the SAME network to the EXACT solution via plain regression
   - If regression succeeds where PINN fails → **optimization failure** (not capacity)
   - If both fail → capacity failure

**Expected finding** (Krishnapriyan et al. 2021): Optimization failure.
The physics loss landscape has pathological gradient flow at high c.

**Report cost alongside accuracy** — always.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# Shared Tools
# ============================================================

class AdvectionPINN(nn.Module):
    """MLP for advection PINN: 4 hidden layers x 64, tanh."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))


def exact_advection(x, t, c, seed=42):
    """Exact advection: u(x,t) = u0((x - c*t) mod 1).
    IC: sum of Fourier modes with k^-2 decay."""
    rng = torch.Generator(); rng.manual_seed(seed)
    x_sh = (x - c * t) % 1.0
    u = torch.zeros_like(x_sh)
    for k in range(1, 17):
        a_k = torch.randn(1, generator=rng).item() * k**-2
        b_k = torch.randn(1, generator=rng).item() * k**-2
        u = u + a_k * torch.sin(2*math.pi*k*x_sh) + b_k * torch.cos(2*math.pi*k*x_sh)
    return u


def rel_l2(pred, exact):
    p = pred.flatten().float(); e = exact.flatten().float()
    return (torch.linalg.norm(p - e) / (torch.linalg.norm(e) + 1e-8)).item()


print('Shared tools ready.')

In [ ]:
# ============================================================
# Part 1: Physics PINN at various speeds
# ============================================================

def train_advection_pinn_physics(c, adam_epochs=2000, n_f=5000):
    """Train advection PINN via physics loss. Returns (L2_error, time, loss_dict)."""
    model = AdvectionPINN().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    t_start = time.time()

    for _ in range(adam_epochs):
        opt.zero_grad()

        x_f = torch.rand(n_f, 1, device=device, requires_grad=True)
        t_f = torch.rand(n_f, 1, device=device, requires_grad=True)
        u   = model(x_f, t_f)
        u_t = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
        u_x = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
        l_res = torch.mean((u_t + c * u_x) ** 2)

        x_ic = torch.rand(n_f//4, 1, device=device)
        t_ic = torch.zeros_like(x_ic)
        u_ic_ex = exact_advection(x_ic, 0.0, c).to(device)
        l_ic = torch.mean((model(x_ic, t_ic) - u_ic_ex) ** 2)

        x_L = torch.zeros(n_f//4, 1, device=device); x_R = torch.ones(n_f//4, 1, device=device)
        t_bc = torch.rand(n_f//4, 1, device=device)
        l_bc = torch.mean((model(x_L, t_bc) - model(x_R, t_bc)) ** 2)

        (l_res + l_ic + l_bc).backward(); opt.step()

    elapsed = time.time() - t_start
    model.eval()
    x_test = torch.linspace(0, 1, 200).view(-1, 1).to(device)
    t_test = torch.ones_like(x_test)
    with torch.no_grad():
        u_pred = model(x_test, t_test)
    u_ex = exact_advection(x_test.cpu(), 1.0, c)
    err = rel_l2(u_pred.cpu(), u_ex)
    return err, elapsed, {'res': l_res.item(), 'ic': l_ic.item(), 'bc': l_bc.item()}


print('=== Part 1: Physics PINN Sweep ===')
c_values = [1, 5, 20, 50]
pinn_results = {}
for c in c_values:
    print(f'  Training PINN for c={c}...')
    err, elapsed, losses = train_advection_pinn_physics(c, adam_epochs=2000)
    pinn_results[c] = {'err': err, 'time': elapsed, 'losses': losses}
    print(f'    c={c:3d}: L2={err:.4f}, time={elapsed:.1f}s, Res={losses["res"]:.3e}')

# Plot the cliff
plt.figure(figsize=(9, 4))
errs = [pinn_results[c]['err'] for c in c_values]
plt.bar([str(c) for c in c_values], errs, color=['green' if e < 0.1 else 'red' for e in errs])
plt.axhline(y=0.1, linestyle='--', color='orange', label='10% threshold')
plt.title('PINN Relative L2 Error vs Advection Speed')
plt.xlabel('Advection speed c'); plt.ylabel('Relative L2 Error')
plt.legend(); plt.grid(True, axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Part 2: Supervised 'Cheat' Test — Diagnosis
# ============================================================
# Key diagnostic: if we give the SAME architecture the EXACT solution as
# supervised targets, can it fit the high-frequency wave?
# YES -> network has capacity -> PINN failure is optimization, not capacity.

class PositionalEncodingPINN(nn.Module):
    """PINN with axis-aligned positional encodings to mitigate spectral bias."""
    def __init__(self, n_freqs=20):
        super().__init__()
        self.freqs = torch.arange(1, n_freqs + 1).float() * math.pi
        in_dim = 2 + 4 * n_freqs  # x, t + sin/cos for each
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1)
        )

    def encode(self, x, t):
        freqs = self.freqs.to(x.device)
        x_enc = torch.cat([torch.sin(f * x) for f in freqs] +
                          [torch.cos(f * x) for f in freqs], dim=-1)
        t_enc = torch.cat([torch.sin(f * t) for f in freqs] +
                          [torch.cos(f * t) for f in freqs], dim=-1)
        return torch.cat([x, t, x_enc, t_enc], dim=-1)

    def forward(self, x, t):
        return self.net(self.encode(x, t))


def train_supervised_cheat(c, n_grid=300, adam_epochs=2500, lbfgs_steps=200):
    """Train the network with exact labels — no physics loss."""
    model = PositionalEncodingPINN().to(device)
    t_start = time.time()

    # Dense grid of (x, t) pairs with exact labels
    x_g = torch.linspace(0, 1, n_grid).to(device)
    t_g = torch.linspace(0, 1, n_grid).to(device)
    XX, TT = torch.meshgrid(x_g, t_g, indexing='ij')
    x_pts = XX.flatten().view(-1, 1)
    t_pts = TT.flatten().view(-1, 1)
    u_ex  = exact_advection(x_pts.cpu(), t_pts.cpu(), c).view(-1, 1).to(device)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(adam_epochs):
        opt.zero_grad()
        u_pred = model(x_pts, t_pts)
        loss = torch.mean((u_pred - u_ex) ** 2)
        loss.backward(); opt.step()
        if ep % 500 == 0:
            print(f'    Adam Epoch {ep}/{adam_epochs} | MSE: {loss.item():.6f}')

    # L-BFGS polish
    lbfgs = optim.LBFGS(model.parameters(), max_iter=lbfgs_steps)
    def closure():
        lbfgs.zero_grad()
        loss = torch.mean((model(x_pts, t_pts) - u_ex) ** 2)
        loss.backward(); return loss
    lbfgs.step(closure)

    elapsed = time.time() - t_start
    model.eval()
    x_test = torch.linspace(0, 1, 200).view(-1, 1).to(device)
    t_test = torch.ones_like(x_test)
    with torch.no_grad():
        u_pred = model(x_test, t_test)
    u_ex_test = exact_advection(x_test.cpu(), 1.0, c)
    err = rel_l2(u_pred.cpu(), u_ex_test)
    return err, elapsed


print('=== Part 2: Supervised Cheat Test (c=50) ===')
print('If this succeeds, PINN failure is optimization, not capacity.')
sup_err_c50, sup_time = train_supervised_cheat(c=50, adam_epochs=2500)
print(f'\nSupervised L2 Error at c=50: {sup_err_c50:.6f}')
pinn_err_c50 = pinn_results.get(50, {}).get('err', float('nan'))
print(f'Physics PINN L2 Error at c=50: {pinn_err_c50:.6f}')

if sup_err_c50 < 0.05:
    print('\n[CONCLUSION] OPTIMIZATION FAILURE CONFIRMED:')
    print('  Network has capacity; physics loss landscape is at fault.')
else:
    print('\n[CONCLUSION] Both fail — capacity is also insufficient at c=50.')

In [ ]:
# ============================================================
# Part 3: Sensitivity / Stability Analysis at c=20
# ============================================================
# Change one variable at a time: loss weighting

def train_weighted_pinn(c, weight_ic=1.0, weight_bc=1.0, epochs=2000, n_f=5000, n_runs=5):
    """Run n_runs independently seeded PINNs and report mean +/- std error."""
    errors = []
    for run in range(n_runs):
        torch.manual_seed(run * 100)
        model = AdvectionPINN().to(device)
        opt   = optim.Adam(model.parameters(), lr=1e-3)
        for _ in range(epochs):
            opt.zero_grad()
            x_f = torch.rand(n_f, 1, device=device, requires_grad=True)
            t_f = torch.rand(n_f, 1, device=device, requires_grad=True)
            u   = model(x_f, t_f)
            u_t = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
            u_x = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
            l_res = torch.mean((u_t + c * u_x) ** 2)
            x_ic = torch.rand(n_f//4, 1, device=device)
            t_ic = torch.zeros_like(x_ic)
            l_ic = torch.mean((model(x_ic, t_ic) - exact_advection(x_ic, 0.0, c).to(device))**2)
            x_L = torch.zeros(n_f//4, 1, device=device); x_R = torch.ones(n_f//4, 1, device=device)
            t_bc = torch.rand(n_f//4, 1, device=device)
            l_bc = torch.mean((model(x_L, t_bc) - model(x_R, t_bc)) ** 2)
            (l_res + weight_ic * l_ic + weight_bc * l_bc).backward(); opt.step()

        model.eval()
        x_test = torch.linspace(0, 1, 200).view(-1, 1).to(device)
        with torch.no_grad():
            u_pred = model(x_test, torch.ones_like(x_test))
        errors.append(rel_l2(u_pred.cpu(), exact_advection(x_test.cpu(), 1.0, c)))
        print(f'    Run {run+1}: L2={errors[-1]:.4f}')

    return np.mean(errors), np.std(errors)


print('=== Part 3: Stability & Loss Weighting (c=20) ===')
print('Config A: Standard weights (IC=1, BC=1)')
mean_A, std_A = train_weighted_pinn(c=20, weight_ic=1.0, weight_bc=1.0, epochs=1500, n_runs=5)
print(f'Config A: L2 = {mean_A:.4f} +/- {std_A:.4f}')

print('\nConfig B: Heavy edge weights (IC=100, BC=100)')
mean_B, std_B = train_weighted_pinn(c=20, weight_ic=100.0, weight_bc=100.0, epochs=1500, n_runs=5)
print(f'Config B: L2 = {mean_B:.4f} +/- {std_B:.4f}')

print('\n=== CONCLUSIONS ===')
if std_A > 0.2:
    print('High variance in Config A: failure is sensitive to initialization.')
if mean_B > 0.5:
    print('Config B also fails: heavy weighting does not cure gradient pathology.')

In [ ]:
# ============================================================
# Final Summary Figure
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Error vs speed (physics PINN)
c_vals = list(pinn_results.keys())
errs   = [pinn_results[c]['err'] for c in c_vals]
times  = [pinn_results[c]['time'] for c in c_vals]

axes[0].bar([str(c) for c in c_vals], errs,
            color=['green' if e < 0.1 else ('orange' if e < 0.5 else 'red') for e in errs])
axes[0].axhline(0.1, linestyle='--', color='gray', label='10% threshold')
axes[0].set_title('Physics PINN: L2 Error vs Advection Speed')
axes[0].set_xlabel('c'); axes[0].set_ylabel('Relative L2')
axes[0].legend(); axes[0].grid(True, axis='y', alpha=0.3)

# Supervised vs physics at c=50
axes[1].bar(
    ['Physics PINN\n(c=50)', 'Supervised\n(c=50)'],
    [pinn_err_c50, sup_err_c50],
    color=['red', 'green']
)
axes[1].set_title('Capacity vs Optimization Diagnosis')
axes[1].set_ylabel('Relative L2 Error')
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('Investigation 1: The Advection Cliff')
plt.tight_layout(); plt.show()

print('\nKey takeaway: PINN fails at high c due to OPTIMIZATION failure,')
print('not because the network lacks capacity to represent the solution.')